In [84]:
import numpy as np
import os
import cv2
import glob
import matplotlib.pyplot as plt
IMG_FOLDER = 'mynteye'

In [85]:
nb_vertical = 9
nb_horizontal = 6
SQUARE = 33.6  # mm

objp = np.zeros((nb_horizontal * nb_vertical, 3), np.float32)
objp[:, :2] = np.mgrid[0:nb_vertical, 0:nb_horizontal].T.reshape(-1, 2) * SQUARE

objpoints, imgpoints_l, imgpoints_r = [], [], []

for fl in sorted(glob.glob(IMG_FOLDER + "/left-*.png")):
    fr = fl.replace("left-", "right-")
    gray_l = cv2.imread(fl, cv2.IMREAD_GRAYSCALE)
    gray_r = cv2.imread(fr, cv2.IMREAD_GRAYSCALE)
    ret_l, c_l = cv2.findChessboardCorners(gray_l, (nb_vertical, nb_horizontal), None)
    ret_r, c_r = cv2.findChessboardCorners(gray_r, (nb_vertical, nb_horizontal), None)
    if ret_l and ret_r:                      # keep only pairs where both work
        objpoints.append(objp)
        imgpoints_l.append(c_l)
        imgpoints_r.append(c_r)
        img = cv2.drawChessboardCorners(gray_l, (nb_vertical, nb_horizontal), c_l, ret_l)
        cv2.imshow('left', img)
        cv2.imshow('right', cv2.drawChessboardCorners(gray_r, (nb_vertical, nb_horizontal), c_r, ret_r))
        cv2.waitKey(10)

gray = gray_l  # used below for image size
print(len(objpoints), "usable pairs")
cv2.destroyAllWindows()

50 usable pairs


In [86]:
size = gray.shape[::-1]
_, K1, D1, _, _ = cv2.calibrateCamera(objpoints, imgpoints_l, size, None, None)
_, K2, D2, _, _ = cv2.calibrateCamera(objpoints, imgpoints_r, size, None, None)
rms, K1, D1, K2, D2, R, T, E, F = cv2.stereoCalibrate(
    objpoints, imgpoints_l, imgpoints_r, K1, D1, K2, D2, size,
    flags=cv2.CALIB_FIX_INTRINSIC)
R1, R2, P1, P2, Q, roi1, roi2 = cv2.stereoRectify(K1, D1, K2, D2, size, R, T, alpha=0)
map_l = cv2.initUndistortRectifyMap(K1, D1, R1, P1, size, cv2.CV_32FC1)
map_r = cv2.initUndistortRectifyMap(K2, D2, R2, P2, size, cv2.CV_32FC1)
print("stereo RMS:", rms)

stereo RMS: 0.8206764037554628


In [87]:
all_dst_l, all_dst_r = [], []
for fl in sorted(glob.glob(IMG_FOLDER + "/left-*.png")):
    fr = fl.replace("left-", "right-")
    all_dst_l.append(cv2.remap(cv2.imread(fl, 0), *map_l, cv2.INTER_LINEAR))
    all_dst_r.append(cv2.remap(cv2.imread(fr, 0), *map_r, cv2.INTER_LINEAR))

In [88]:
from numpy.lib.stride_tricks import sliding_window_view

def compute_disparity_map(left_gray, right_gray, block_size=7, max_disparity=64):
    """
    For each block in left_gray, search the same row in right_gray for the
    best SAD match. This uses the rectified stereo assumption, the match
    is always on the same row.

    The search only checks x' in [x - max_disparity, x], since a matching
    point in the right image only ever shifts left compared to the left
    image.

    max_disparity limits how far we search. Without it, we would compare
    every template against every position in the row, which is slow and
    not needed here, since this scene's true disparities stay small.

    Computationally less expensive as the nested for loops are omitted.
    """
    half = block_size // 2
    h, w = left_gray.shape
    disparity_map = np.zeros((h, w), dtype=np.float32)

    for y in range(half, h - half):
        # All possible block_size x block_size windows along this row of the
        # right image, computed once per row.
        right_row = right_gray[y-half:y+half+1, :].astype(np.int32)
        right_windows = sliding_window_view(right_row, (block_size, block_size))[0]
        # right_windows[i] = the window whose left edge is at column i

        for x in range(half, w - half):
            template = left_gray[y-half:y+half+1, x-half:x+half+1].astype(np.int32)

            search_lo = max(0, (x - half) - max_disparity)
            search_hi = (x - half) + 1  # candidates with left edge <= template's left edge
            candidates = right_windows[search_lo:search_hi]

            diffs = np.abs(candidates - template)
            scores = diffs.sum(axis=(1, 2))          # SAD per candidate
            best_local = np.argmin(scores)
            best_left_edge = search_lo + best_local

            disparity_map[y, x] = (x - half) - best_left_edge

    return disparity_map

os.makedirs(f'disparity_maps/{IMG_FOLDER}', exist_ok=True)
all_disparity_map = []
for i in range(len(all_dst_l)):
    disparity_map = compute_disparity_map(all_dst_l[i], all_dst_r[i], block_size=15, max_disparity=64)
    all_disparity_map.append(disparity_map)
    print(f'finished disparity map {i+1} of {len(all_dst_l)}')
    plt.imsave(f'disparity_maps/{IMG_FOLDER}/disparity_map_{i:02d}.png', disparity_map)


finished disparity map 1 of 50
finished disparity map 2 of 50
finished disparity map 3 of 50
finished disparity map 4 of 50


KeyboardInterrupt: 